<a href="https://colab.research.google.com/github/virajsharma2000/Photolab/blob/master/demo_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Visualization Demo

This notebook demonstrates how to use the this library to visualize the internal activations of a quantized GPT-2 model, just create a `intrav_viz` directory on colab and upload the python files init, hook_manager and visualizer.
Requirements: Make sure the `intra_viz` folder is uploaded to the same directory as this notebook.

In [3]:
# Clone the repository
!git clone https://github.com/modelrecon/mr-viz-experiments.git
# Move into the repo directory
!pip install transformers accelerate bitsandbytes plotly
import sys
sys.path.append('/content/mr-viz-experiments')

Cloning into 'mr-viz-experiments'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 55 (delta 29), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 23.81 KiB | 840.00 KiB/s, done.
Resolving deltas: 100% (29/29), done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# Import directly from the uploaded package
from visualizer import TransformerVisualizer

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Load Quantized Model (4-bit)
model_id = "gpt2"

if device == "cuda":
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config)
else:
    # Fallback for CPU (no bitsandbytes 4bit support on CPU usually)
    print("Quantization not supported on CPU, loading standard model.")
    model = AutoModelForCausalLM.from_pretrained(model_id)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Initialize Visualizer
visualizer = TransformerVisualizer(model, tokenizer, device=device)

In [7]:
# Generate Visualization
input_text = "The quick brown fox jumps over the lazy dog"
fig = visualizer.visualize(input_text, top_k=10)

# Display
fig.show()

Model Architecture Scan:
Hooking layers: ['transformer.h.0', 'transformer.h.1', 'transformer.h.2', 'transformer.h.3', 'transformer.h.4', 'transformer.h.5', 'transformer.h.6', 'transformer.h.7', 'transformer.h.8', 'transformer.h.9', 'transformer.h.10', 'transformer.h.11']
